## Evaluation of Words as Frames

The Frame Representation Hypothesis assumes we can build concepts by "averaging" word frames.

One way to verify this claim is by checking if the Concept Frame correlation with the words used
to generate is higher than its correlation with other words or random frames.

In [1]:
import pandas as pd
import torch
from matplotlib import pyplot as plt

from frames.representations import FrameUnembeddingRepresentation
from frames.utils.memory import garbage_collection_cuda
from frames.utils.plotting import histplot_and_save
from frames.utils.settings import load_models

In [ ]:
MODELS = load_models()

LEMMA_COUNT = {
    "gemma": 22,
    "llama": 54,
    "phi": 15,
}  # avoid GPU OOM errors (+80GB required)
FRAME_RANK = 3
N_RANDOM = 100

X = "total projection"
HUE = "model family"

RAND_PROJ = "Random Word Projection"

In [3]:
def compute_similarities(concepts, words):
    sim = concepts.similarity(words)
    return sim[sim != 0]  # remove padded null vectors and flatten


def compare_words_and_random_frames(*args, **kwargs):
    garbage_collection_cuda()

    fur = FrameUnembeddingRepresentation.from_model_id(*args, **kwargs)

    kw = dict(
        min_lemmas_per_synset=next(
            v for k, v in LEMMA_COUNT.items() if k in kwargs["id"].lower()
        ),
        max_token_count=FRAME_RANK,
    )

    all_concepts = fur.get_all_concepts(**kw)
    print("Concepts =", all_concepts)

    all_word_frames = fur.get_all_words_frames(**kw)
    print("Words =", all_word_frames)

    # n = all_word_frames.tensor.size(1)
    # random_indices = torch.randperm(n)[:N_RANDOM]
    # random_word_frames = all_word_frames.tensor[0, random_indices].unsqueeze(1).repeat(1, n, 1, 1)
    random_word_frames = torch.rand_like(all_word_frames.tensor[:N_RANDOM])
    print("Random Words =", random_word_frames.shape)

    concept_word_sim = compute_similarities(all_concepts, all_word_frames).float()
    concept_random_sim = compute_similarities(all_concepts, random_word_frames).float()

    rand_df = pd.DataFrame({X: concept_random_sim.cpu()}).assign(**{HUE: RAND_PROJ})
    word_df = pd.DataFrame({X: concept_word_sim.cpu()}).assign(**{HUE: str(fur.model.family)})

    return pd.concat([rand_df, word_df]).dropna()


df = pd.concat(
    [
        compare_words_and_random_frames(**kwargs)
        for kwargs in MODELS.to_dict(orient="index").values()
    ]
)

plt.yscale("log")
histplot_and_save("03_concept_vs_word_frames_relationship", df, X, HUE, discrete=True)

Token has not been saved to git credential helper.


Cannot authenticate through git-credential as no helper is defined on your machine.
You might have to re-authenticate when pushing to the Hugging Face Hub.
Run the following command in your terminal in case you want to set the 'store' credential helper as default.

git config --global credential.helper store

Read https://git-scm.com/book/en/v2/Git-Tools-Credential-Storage for more details.


/home/rodrigo/projects/frh-multi/.venv/lib/python3.11/site-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

2026-07-03 17:16:38.101 | INFO     | frames.models.hf.base:__init__:88 - Loaded model: hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4
2026-07-03 17:16:38.104 | WARNING  | frames.models.hf.base:__init__:89 - memory cost: 5462 Mb


Concepts = Concept(count=946, dimension=4096, rank=3)
Words = Frame(shape=torch.Size([271, 946, 4096, 3]))
Random Words = torch.Size([100, 946, 4096, 3])


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.87 GiB. GPU 2 has a total capacity of 23.55 GiB of which 5.58 GiB is free. Including non-PyTorch memory, this process has 17.94 GiB memory in use. Of the allocated memory 16.61 GiB is allocated by PyTorch, and 1.05 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)